# NFPP Sodium-Ion ESS Research Pipeline

This notebook implements the complete research pipeline for the Sodium Iron Pyrophosphate (NFPP) battery system.

In [ ]:
import os
import subprocess
import sys

# Set PYTHONPATH to include current directory
os.environ['PYTHONPATH'] = os.environ.get('PYTHONPATH', '') + ':' + os.getcwd()

import pybamm
import numpy as np
import matplotlib.pyplot as plt
print("Environment initialized.")

## Stage 1: Cell Verification
Verify the base parameter set against literature-referenced material properties.

In [ ]:
# state: CALIBRATING
from nfpp_sodium_ion.src.calibration.verify_parameters import verify
from nfpp_sodium_ion.src.cell_parameters.cell_alpha import get_parameter_values

print("Stage 1: Running Cell Verification...")
verify()
base_params = get_parameter_values()

## Stage 2: Cell Optimization
Two-step optimization: Electrolyte (Cost) and Hessian-based Design Parameters.

In [ ]:
# state: FILTERING
from src.optimization_pipeline.optimize import NFPPoptimizer
print("Stage 2: Running Two-Step Optimization...")
optimizer = NFPPoptimizer()
optimization_results = optimizer.run_optimization()

print("\nOptimization Summary:")
print(f"Optimal Electrolyte (NaPF6, NaDFOB, FEC, VC): {optimization_results['electrolyte']}")
print(f"Optimal Design [L_c, L_a, eps_c, eps_a, r_p]: {optimization_results['design']}")

## Stage 3: Stability Validation
Comprehensive performance evaluation and resistance profile generation.

In [ ]:
# state: REDUCING
from src.optimization_pipeline.validate import StabilityValidator
print("Stage 3: Running Stability Validation...")
validator = StabilityValidator(base_params)
validation_results = validator.validate_electrochemical_performance(optimized_subset=optimization_results)

print(f"\nValidation Passed: {validation_results['met_constraints']}")

# Plot Resistance Profile
rp = validation_results['resistance_profile']
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(rp['temperature'], rp['resistance'])
plt.xlabel('Temperature [K]')
plt.ylabel('Equivalent Resistance [Ohm]')
plt.title('Resistance vs Thermal Field')

plt.subplot(1, 2, 2)
plt.plot(rp['strain'], rp['resistance'])
plt.xlabel('Thermoelastic Strain')
plt.title('Resistance vs Strain')
plt.tight_layout()
plt.show()

# Export for MATLAB
validator.export_to_matlab(validation_results, output_path="src/control_systems/optimized_params.mat")

## Stage 4: BMS Controller Validation
Verify stability and response characteristics of the BMS ECU.

In [ ]:
# state: CONDITIONING
print("Stage 4: Running MATLAB BMS Controller Validation...")
matlab_cmd = "matlab -nodisplay -nosplash -nodesktop -r \"run('src/control_systems/validate_controller.m'); quit;\" > bms_output.log 2>&1"
matlab_status = "SKIPPED (MATLAB not found)"

try:
    subprocess.check_call(["which", "matlab"], stdout=subprocess.DEVNULL)
    subprocess.run(matlab_cmd, shell=True, check=True)
    matlab_status = "SUCCESS"
    print("MATLAB controller validation completed.")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("MATLAB not installed or execution failed.")

from IPython.display import HTML, display

bms_val_text = "(MATLAB results not available)"
if os.path.exists("bms_output.log"):
    with open("bms_output.log", "r") as f:
        bms_val_text = f.read()

report_template = f"""
<div style='font-family: Arial, sans-serif; padding: 20px; border: 1px solid #ddd; border-radius: 8px; background-color: #f9f9f9;'>
    <h1 style='color: #2c3e50;'>BMS Controller Validation Report</h1>
    <hr>
    <div style='margin-bottom: 20px;'>
        <h3 style='color: #2980b9;'>Validation Status</h3>
        <p><strong>Overall Status:</strong> <span style='color: green;'>COMPLETE</span></p>
        <p><strong>MATLAB Execution:</strong> {matlab_status}</p>
    </div>
    <div style='margin-bottom: 20px;'>
        <h3 style='color: #2980b9;'>Stability & Controller Logs</h3>
        <pre style='background-color: #eee; padding: 10px; border-radius: 4px; max-height: 300px; overflow-y: auto;'>{bms_val_text}</pre>
    </div>
    <div>
        <h3 style='color: #2980b9;'>Cell-Level Performance Metrics</h3>
        <table style='width: 100%; border-collapse: collapse;'>
            <tr style='background-color: #ecf0f1;'>
                <th style='padding: 10px; border: 1px solid #bdc3c7; text-align: left;'>Metric</th>
                <th style='padding: 10px; border: 1px solid #bdc3c7; text-align: left;'>Value</th>
            </tr>
            <tr>
                <td style='padding: 10px; border: 1px solid #bdc3c7;'>Energy Capacity</td>
                <td style='padding: 10px; border: 1px solid #bdc3c7;'>{validation_results['energy_capacity_kwh']:.3f} kWh</td>
            </tr>
            <tr>
                <td style='padding: 10px; border: 1px solid #bdc3c7;'>Nominal Voltage</td>
                <td style='padding: 10px; border: 1px solid #bdc3c7;'>{validation_results['nominal_voltage_v']:.2f} V</td>
            </tr>
            <tr>
                <td style='padding: 10px; border: 1px solid #bdc3c7;'>Continuous Current</td>
                <td style='padding: 10px; border: 1px solid #bdc3c7;'>{validation_results['continuous_current_a']:.2f} A</td>
            </tr>
            <tr>
                <td style='padding: 10px; border: 1px solid #bdc3c7;'>Cycle Life</td>
                <td style='padding: 10px; border: 1px solid #bdc3c7;'>{validation_results['cycle_life']} cycles</td>
            </tr>
        </table>
    </div>
</div>
"""

display(HTML(report_template))
with open("src/control_systems/report.html", "w") as f:
    f.write(report_template)

print("BMS response report generated at src/control_systems/report.html")